In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sqlite3

# Task 0
Data extraction: get the data from 3 tables & combine it into single `.csv` file.
After that read this file using pandas to create Dataframe.
So it will be all joined data in 1 dataframe. Quick check - should be 74818 rows in it.

In [ ]:
conn = sqlite3.connect("../db.sqlite3")
cur = conn.cursor()
data = pd.read_sql_query(
    "SELECT restaurant_order.id, quantity, datetime, price, name FROM restaurant_orderitem LEFT JOIN restaurant_order on restaurant_orderitem.order_id = restaurant_order.id LEFT JOIN restaurant_product on restaurant_orderitem.product_id = restaurant_product.id",
    conn)
data


# Task 1
Get Top 10 most popular products in restaurant sold by Quantity.
Count how many times each product was sold and create a pie chart with percentage of popularity (by quantity) for top 10 of them.

Example:

![pie chart](../demo/pie.png)

In [ ]:
df_analysis = data[['name', 'quantity']].copy()
df_analysis['quantity'] = pd.to_numeric(df_analysis['quantity'], errors='coerce')
df_analysis.dropna(subset=['quantity'], inplace=True)

product_popularity = df_analysis.groupby('name')['quantity'].sum().reset_index()
top_ten = product_popularity.sort_values(by='quantity', ascending=False).head(10)

total_quantity = top_ten['quantity'].sum()
top_ten['percentage'] = (top_ten['quantity'] / total_quantity) * 100
plt.pie(top_ten['percentage'], labels=top_ten['name'], autopct=f'%1.1f%%')
plt.title("Top 10 positions in menu by quantity")
plt.ylabel("Quantity")


# Task 2
Calculate `Item Price` (Product Price * Quantity) for each Order Item in dataframe.
And Make the same Top 10 pie chart, but this time by `Item Price`. So this chart should describe not the most popular products by quantity, but which products (top 10) make the most money for restaurant. It should be also with percentage.

In [ ]:
df_analysis = data[['name','id', "quantity", "price"]].copy()
df_analysis['item_price'] = df_analysis['price'] * df_analysis['quantity']
top_ten = (
    df_analysis.groupby('name')['item_price']
    .sum()
    .reset_index()
).sort_values(by='item_price', ascending=False).head(10)
total_sum = top_ten['item_price'].sum()
top_ten['percentage'] = (top_ten['item_price'] / total_sum) * 100
plt.pie(top_ten['percentage'], labels=top_ten['name'], autopct=f'%1.1f%%')
plt.title("Top 10 Products by Revenue")
plt.show()

# Task 3
Calculate `Order Hour` based on `Order Datetime`, which will tell about the specific our the order was created (from 0 to 23). Using `Order Hour` create a bar chart, which will tell the total restaurant income based on the hour order was created. So on x-axis - it will be values from 0 to 23 (hours), on y-axis - it will be the total sum of order prices, which were sold on that hour.

Example:

![bar chart](../demo/bar.png)

In [ ]:
df_analysis = data[['datetime', 'id', "quantity", "price"]].copy()
df_analysis['datetime'] = pd.to_datetime(df_analysis['datetime'], errors='coerce')
df_analysis['order_hour'] = df_analysis['datetime'].dt.hour
df_analysis['item_price'] = df_analysis['price'] * df_analysis['quantity']
hourly = df_analysis.groupby('order_hour')['item_price'].sum().reset_index().sort_values(by='item_price', ascending=False)
plt.figure(figsize=(12, 6))
plt.bar(hourly['order_hour'], hourly['item_price'])
plt.xlabel("Order Hour")
plt.title("Profit By Order Hour")
plt.xticks(range(0, 24))
plt.grid(axis='y', linestyle='--', linewidth=0.5, alpha=0.7)
plt.show()

# Task 4
Make similar bar chart, but right now with `Order Day Of The Week` (from Monday to Sunday), and also analyze total restaurant income by each day of the week.

In [ ]:
df_analysis = data[['datetime', 'id', "quantity", "price"]].copy()
df_analysis['datetime'] = pd.to_datetime(df_analysis['datetime'], errors='coerce')
df_analysis['order_day'] = df_analysis['datetime'].dt.day_name()
df_analysis['order_day_index'] = df_analysis['datetime'].dt.dayofweek
df_analysis['item_price'] = df_analysis['price'] * df_analysis['quantity']
daily = df_analysis.groupby(['order_day', 'order_day_index'])['item_price'].sum().reset_index().sort_values(by='order_day_index', ascending=True)
plt.figure(figsize=(12, 6))
plt.bar(daily['order_day'], daily['item_price'])
plt.xlabel("Order Day")
plt.title("Profit By Order Hour")
plt.xticks(range(0, 7))
plt.grid(axis='y', linestyle='--', linewidth=0.5, alpha=0.7)
plt.show()